# Auto-Grading Algorithm for Fischer Projections

This notebook demonstrates the grading algorithm for **Fischer projections** used in the StereoLearn platform, with the original PHP logic transcribed into notebook format.

The grading workflow consists of three steps:
| Step | Description |
|------|-------------|
| **Step 0** | Compare non-isomeric canonical SMILES — verify chemical identity |
| **Step 1** | Parse V2000 Molfiles and extract Fischer projection features |
| **Step 2** | Compare features of the key and the input to determine correctness |

**Example molecule**: D-glucose (Fischer projection)

In [ ]:
import re
import os
from rdkit import Chem
# ---------------------------------------------------------------------------
# Step 0: Compare non-isomeric canonical SMILES
# ---------------------------------------------------------------------------
def mol_to_smiles(mol_file_path):
    """Convert a V2000 Molfile to a non-isomeric canonical SMILES string."""
    try:
        mol = Chem.MolFromMolFile(mol_file_path)
        if mol is None:
            raise ValueError("Error: Check file path and format.")
        mol = Chem.RemoveHs(mol)
        return Chem.MolToSmiles(mol, isomericSmiles=False, canonical=True)
    except Exception as e:
        print(f"Error when processing Molfile: {e}")
        return None

# Answer key Molfile path here
key_mol_file = "D-glucose_key.mol" #use D-glucose as an example
key_canonical_smiles = mol_to_smiles(key_mol_file)

# Student input
input_mol_file = "D-glucose_input1.mol"       # input1 — correct
#input_mol_file = "D-glucose_input2.mol"     # input2 — wrong (stereoisomer)

input_smiles = mol_to_smiles(input_mol_file)

mark_smiles = 1 if input_smiles == key_canonical_smiles else 0
print("non-isomeric SMILES match?", mark_smiles)

# ---------------------------------------------------------------------------
# Step 1: Parse Molfiles and extract Fischer projection features
# ---------------------------------------------------------------------------
def normalize_spaces(s):
    """Collapse multiple spaces into one."""
    return re.sub(r' {2,}', ' ', s).strip()

def parse_mol_atoms(file_path):
    """Parse the atom block of a V2000 Molfile.
    Returns list of [x, y, element, original_index]."""
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    try:
        parts = normalize_spaces(lines[3]).split(' ')
        n_atoms = int(parts[0])
        n_bonds = int(parts[1])
    except (IndexError, ValueError):
        raise ValueError("Invalid Molfile format")

    atoms = []
    for idx, i in enumerate(range(4, 4 + n_atoms), start=1):
        tmp = normalize_spaces(lines[i]).split(' ')
        atoms.append([float(tmp[0]), float(tmp[1]), tmp[3], idx])

    # parse bond block for adjacency
    bond_start = 4 + n_atoms
    link = {}
    for i in range(bond_start, bond_start + n_bonds):
        tmp = normalize_spaces(lines[i]).split(' ')
        a1, a2 = int(tmp[0]), int(tmp[1])
        link.setdefault(a1, []).append(a2)
        link.setdefault(a2, []).append(a1)

    return atoms, link

def extract_fischer_feature(mol_file_path):
    """Extract the F1-F2-F3 Fischer projection feature from a Molfile."""
    # F1: SMILES
    mol = Chem.MolFromMolFile(mol_file_path)
    if mol is None:
        raise ValueError(f"Cannot parse Molfile: {mol_file_path}")
    mol = Chem.RemoveHs(mol)
    f1 = Chem.MolToSmiles(mol, isomericSmiles=True)

    atoms, link = parse_mol_atoms(mol_file_path)

    # sort by y ascending, then by element
    atoms.sort(key=lambda a: (a[1], a[2]))

    # find first and last C atoms
    first_c = None
    last_c = None
    for p in atoms:
        if p[2] == 'C':
            if first_c is None:
                first_c = p
            last_c = p

    # collect terminal neighbors to remove (keep C atoms)
    del_nodes = set()
    for neighbor in link.get(first_c[3], []):
        del_nodes.add(neighbor)
    for neighbor in link.get(last_c[3], []):
        del_nodes.add(neighbor)

    atoms = [p for p in atoms if (p[3] not in del_nodes) or (p[2] == 'C')]

    # F3: vertical element sequence (top to bottom, skip H)
    sindex = None
    eindex = None
    f3 = ''
    for ci, p in enumerate(atoms):
        if p[2] != 'H':
            f3 += p[2]
        if p[2] == 'C':
            if sindex is None:
                sindex = ci
            eindex = ci

    # F2: horizontal element sequence (left to right, between first and last C, skip H)
    xa = [p for ci, p in enumerate(atoms) if sindex <= ci <= eindex]
    xa.sort(key=lambda a: (a[0], a[2]))
    f2 = ''.join(p[2] for p in xa if p[2] != 'H')

    return f"{f1}-{f2}-{f3}"

# ---------------------------------------------------------------------------
# Step 2: Extract features for key and input, then compare
# ---------------------------------------------------------------------------

key_feature = extract_fischer_feature(key_mol_file)
print("key feature:", key_feature)

input_feature = extract_fischer_feature(input_mol_file)
print("input feature:", input_feature)

mark = 1 if input_feature == key_feature else 0
print("final mark is (correct=1, wrong=0):", mark)

**Cite this work**

> Li, X.; Hu, C.; Su, Y.; Lu, H. "StereoLearn: An Online Platform for Learning and Automatic Grading of Organic Structure Representations." *Journal of Chemical Education*, 2026.

If you use StereoLearn and associated content in your teaching or research, please cite the above publication.

📧 Contact: xiaoli@pku.edu.cn

Thank you!